# Speaker Diarization: me vs. someone else

## 1. Clean my recording

Raw video → 16 kHz mono WAV → cut silence with VAD → drop bad chunks → normalize loudness → time-based train/val split.

In [1]:
import subprocess
from pathlib import Path

import numpy as np
import soundfile as sf
import torch
from silero_vad import get_speech_timestamps, load_silero_vad

SR = 16000
RAW = Path("150 turkish text reading.mp4")
DATA = Path("data")
WAV = DATA / "me_16k.wav"
OUT = DATA / "me"
OUT.mkdir(parents=True, exist_ok=True)

In [2]:
# pyannote and silero both expect 16 kHz mono
if not WAV.exists():
    subprocess.run(["ffmpeg", "-v", "error", "-y", "-i", str(RAW), "-vn",
                    "-ac", "1", "-ar", str(SR), "-c:a", "pcm_s16le", str(WAV)], check=True)

audio, sr = sf.read(WAV, dtype="float32")
assert sr == SR and audio.ndim == 1
print(f"{len(audio) / SR / 60:.1f} min, peak {np.abs(audio).max():.2f}")

13.0 min, peak 0.85


### Cut silence

Silence labelled as "me" would teach the model that silence is my voice, so only speech is kept. Segments are capped at 10 s so they can be shuffled into fake conversations later.

In [3]:
vad = load_silero_vad()
segments = get_speech_timestamps(
    torch.from_numpy(audio), vad, sampling_rate=SR,
    min_silence_duration_ms=300, max_speech_duration_s=10, speech_pad_ms=50,
)
speech_s = sum(s["end"] - s["start"] for s in segments) / SR
print(f"{len(segments)} segments, {speech_s / 60:.1f} min of speech ({speech_s / (len(audio) / SR):.0%} of the recording)")

/home/darkwarro/Documents/Web_Apps/Exposure Academy/Academy_Exercises/Speaker_Diarization/.venv/lib/python3.12/site-packages/torch/jit/_serialization.py:176: FutureWarning: `torch.jit.load` is deprecated. Please switch to `torch.export`.
  warnings.warn(


162 segments, 9.0 min of speech (69% of the recording)


### Drop bad chunks, normalize, save

Too short (< 1 s) or clipped chunks are dropped. The rest are scaled to the same loudness (-23 dBFS RMS). No denoising: the real mic will be noisy too.

In [4]:
TARGET_RMS = 10 ** (-23 / 20)

for f in OUT.glob("*.wav"):
    f.unlink()

rows, dropped = [], {"short": 0, "clipped": 0}
for seg in segments:
    chunk = audio[seg["start"]:seg["end"]]
    if len(chunk) < SR:
        dropped["short"] += 1
        continue
    if np.mean(np.abs(chunk) > 0.99) > 0.001:
        dropped["clipped"] += 1
        continue
    chunk = chunk * (TARGET_RMS / np.sqrt(np.mean(chunk ** 2)))
    chunk = chunk / max(1.0, np.abs(chunk).max() / 0.99)  # never clip after gain
    name = f"me_{len(rows):04d}.wav"
    sf.write(OUT / name, chunk, SR, subtype="PCM_16")
    rows.append((name, seg["start"] / SR, seg["end"] / SR))

durs = np.array([e - s for _, s, e in rows])
print(f"kept {len(rows)} chunks, {durs.sum() / 60:.1f} min | dropped {dropped}")
print(f"chunk length: min {durs.min():.1f}s, median {np.median(durs):.1f}s, max {durs.max():.1f}s")

kept 162 chunks, 9.0 min | dropped {'short': 0, 'clipped': 0}
chunk length: min 1.3s, median 3.2s, max 6.7s


### Train / validation split

Split by time, not randomly: the last 15 % of the recording is validation, so no sentence leaks into both sets.

In [5]:
import csv

cut = np.searchsorted(np.cumsum(durs), 0.85 * durs.sum())
with open(DATA / "me_manifest.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["file", "start", "end", "split"])
    for i, (name, s, e) in enumerate(rows):
        w.writerow([name, f"{s:.2f}", f"{e:.2f}", "train" if i < cut else "val"])

assert 0 < cut < len(rows)
assert rows[cut - 1][2] <= rows[cut][1], "train must end before val starts"
print(f"train {durs[:cut].sum() / 60:.1f} min ({cut} chunks) | val {durs[cut:].sum() / 60:.1f} min ({len(rows) - cut} chunks)")

train 7.7 min (135 chunks) | val 1.4 min (27 chunks)
